# Forecasts

This note book can be run all at once. Do not attempt to run anything out of order.

**Summary of what is Forecasted and Models Chosen:**

a. The Citywide Forecast uses the NeuralProphet model.

b. The Manhattan Forecast uses the NeuralProphet model.

c. The Brooklyn Forecast uses the NeuralProphet model.

d. The Daily Staten Island Forecast uses the Prophet model. The Weekly Staten Island Forecast uses the Seasonal Average Baseline Model.

e. The Queens Forecast uses the Prophet model. Even though the notebbok bronx_and_queens.ipynb indicated that the ensemble model which took the average between prophet, ridge regression, xgboost, and SARIMA performed better, the improvement was minimal and the extra computational effort to fit an XGBoost to the entire dataset pushed us to use Prophet.

f. The Bronx Forecast uses the Prophet model.

**Explanation of the Table:** At the end of this notebook, one sees a displayed table of the forecasted number of 311 rat sightings based on the models chosen. Note, however, that if one adds up the forecasted rat sightings in each borough, one *does not always get that it equals the forecasted number of rat sightings citywide*. This is *not a mistake*! We have models for each borough and a separate model for citywide rat sightings. Each model originally makes a forecast which originally need not be an integer. For the purposes of making an integer-valeud forecast, we round the forecasted number to the nearest integer. Due to rounding errors and the citywide modeling being done separately, one gets this seemingly odd discrepency. For this reason, there is a column titled "Summed Citywide Forecast" which simply sums up each borough's forecast.

## User defined input

In [48]:
# STARTING_DATE = pd.datetime(today)
forecast_horizon = 14

## Import and Download Data

In [49]:
# load packages

import requests 
import random
import torch
import numpy as np
import pandas as pd
import warnings
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)
from neuralprophet import NeuralProphet
import logging

from prophet import Prophet

import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

In [50]:
url = "https://data.cityofnewyork.us/api/views/3q43-55fe/rows.csv?accessType=DOWNLOAD"

response = requests.get(url)
response.raise_for_status()

with open("../scr/data/rat_sightings_data/Updated_Rat_Sightings_NYC.csv", "wb") as f:
    f.write(response.content)

## Citywide

In [51]:
params = dict()
params['apparent_temperature_max'] = 30
params['apparent_temperature_min'] = 5
params['snowfall_sum'] = 1
params['n_lags'] = 16
params['epochs'] = 60 
params['learning_rate'] = 0.2986532324899507
params['batch_size'] = 128 
params['ar_reg'] = 2.132925719127823
params['n_forecasts'] = 14 

In [52]:
rat_sighting = pd.read_csv("../scr/data/rat_sightings_data/Updated_Rat_Sightings_NYC.csv")
rat_sighting.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rat_sighting.columns] #apply to column headers
rat_sighting['location_type'] = rat_sighting['location_type'].str.strip().str.replace(' ', '_').str.lower()  #apply to location_type column
cols_to_drop = [c for c in rat_sighting.columns if (rat_sighting[c].nunique(dropna=False) == 1)]
rat_sighting = rat_sighting.drop(columns=cols_to_drop)
rat_sighting['created_date'] = pd.to_datetime(rat_sighting['created_date']) 
rat_sighting = rat_sighting.drop(columns='park_borough')
rat_sighting = rat_sighting.drop(columns=['location'])
rs = rat_sighting.copy()
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

WARNING - (py.warnings._showwarnmsg) - /var/folders/ry/m6r2ndwd10bdv8tvww5hr2680000gn/T/ipykernel_97423/3890385839.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  rat_sighting['created_date'] = pd.to_datetime(rat_sighting['created_date'])



In [53]:
today = rs['ds'].max()
last_date = today


In [54]:
# Weather data
# The forecasted weather values are the naive forecast i.e. we take the last observed day and assume that these are good predictions for the next 14 days.
# Better weather data might help improve the model.

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = last_date.strftime("%Y-%m-%d")  # use last date of rs

url = ("https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd['ds'] = pd.to_datetime(nd['date'])
    wd = nd.drop(columns=['date'])
else:
    wd = pd.DataFrame(data["daily"])
    wd["ds"] = pd.to_datetime(wd["time"])
    wd = wd.drop(columns=["time"])

wd = wd.reset_index(drop=True)
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                             periods=14,
                             freq='D')

last_row = wd.iloc[-1]
wd_14 = pd.DataFrame([last_row.values] * 14, columns=wd.columns)
wd_14['ds'] = future_dates 
wd_14 = wd_14.reset_index(drop=True)

# # weather data in wd_14 should be the same as the last row of wd. 
# # check the displayed dataframe that this is the case

# display(wd.tail(2))
# display(wd_14)


In [55]:
# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

In [56]:
# set-up regressed features and final set-up for forecasting

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']
wd["ds"] = pd.to_datetime(wd["ds"])
wd_14["ds"] = pd.to_datetime(wd_14["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])
rs = rs.merge(wd[['ds'] + regressed_features], on="ds", how="left")

In [57]:
# Fix the seed for the model
# set seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# make PyTorch deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [58]:
forecast_horizon = 14
regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

model = NeuralProphet(yearly_seasonality=True, 
                      weekly_seasonality=True, 
                      epochs=params['epochs'], 
                      learning_rate=params['learning_rate'], 
                      batch_size=params['batch_size'], 
                      n_forecasts = params['n_forecasts'], 
                      n_lags = params['n_lags'], 
                      ar_reg = params['ar_reg'], 
                      )
model = model.add_country_holidays(country_name="US")

for col in regressed_features:
    model.add_lagged_regressor(col, n_lags=params[col])

model.fit(rs, freq="D")
future = model.make_future_dataframe(rs, periods=forecast_horizon, n_historic_predictions=28)

for col in regressed_features:
    future[col] = pd.concat([rs[col], wd_14[col]], ignore_index=True)

forecast = model.predict(future)

rs_df = model.get_latest_forecast(forecast, include_previous_forecasts = 14)

WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(



Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/neuralprophet/data/split.py:273: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, future_df])

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 98.611% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 98.61

Predicting: 18it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


In [59]:
df_wide= rs_df.copy()

# use the last 14 entries of 'origin-0' as the forecast
forecast_horizon = 14
origin_col = 'origin-0' 
last_14_preds = df_wide[origin_col].dropna().tail(forecast_horizon)

# compute corresponding forecast dates
last_14_dates = pd.to_datetime(df_wide['ds'].tail(forecast_horizon))

citywide_forecast = pd.DataFrame({'ds': last_14_dates, 
                                  'yhat': np.clip(np.round(last_14_preds.astype(float)), 0, None).astype(int)})

In [60]:
citywide_forecast

,ds,yhat
14,2026-03-14,14
15,2026-03-15,23
16,2026-03-16,48
17,2026-03-17,44
18,2026-03-18,44
19,2026-03-19,42
20,2026-03-20,35
21,2026-03-21,18
22,2026-03-22,22
23,2026-03-23,49


## Manhattan

In [61]:
params = dict()
params['apparent_temperature_max'] = 54
params['apparent_temperature_min'] = 18
params['snowfall_sum'] = 1
params['n_lags'] = 59 
params['epochs'] = 493 
params['learning_rate'] = 0.003214767890388168 
params['batch_size'] = 220
params['ar_reg'] = 0.5847571241076923
params['n_forecasts'] = 14 

In [62]:
rs = pd.read_csv('../scr/data/rat_sightings_data/Updated_Rat_Sightings_NYC.csv')
rs.columns
rs = rs[['Borough', 'Created Date']]


rs.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rs.columns] #apply to column headers
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']

rs = rs[rs['borough']=='MANHATTAN']



rs['created_date'] = pd.to_datetime(rs['created_date'])
rs['ds'] = rs['created_date'].dt.date
rs = rs.groupby('ds').size().reset_index(name='y')
rs['ds'] = pd.to_datetime(rs['ds'])

full_range = pd.date_range(start='2020-01-01', end=today)

# Reindex to include all dates and fill missing y with 0
rs = rs.set_index('ds').reindex(full_range, fill_value=0).rename_axis('ds').reset_index()

WARNING - (py.warnings._showwarnmsg) - /var/folders/ry/m6r2ndwd10bdv8tvww5hr2680000gn/T/ipykernel_97423/2337001431.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  rs['created_date'] = pd.to_datetime(rs['created_date'])



In [63]:
# Weather data

lat, lon = 40.7831, -73.9712
last_date = today
start = "2020-01-01"
end   = last_date.strftime("%Y-%m-%d")  # use last date of rs

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd['ds'] = pd.to_datetime(nd['date'])
    wd = nd.drop(columns=['date'])
else:
    wd = pd.DataFrame(data["daily"])
    wd["ds"] = pd.to_datetime(wd["time"])
    wd = wd.drop(columns=["time"])

wd = wd.reset_index(drop=True)
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                             periods=14,
                             freq='D')

last_row = wd.iloc[-1]
wd_14 = pd.DataFrame([last_row.values] * 14, columns=wd.columns)
wd_14['ds'] = future_dates 
wd_14 = wd_14.reset_index(drop=True)

# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

# set-up regressed features and final set-up for forecasting

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']
wd["ds"] = pd.to_datetime(wd["ds"])
wd_14["ds"] = pd.to_datetime(wd_14["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])
rs = rs.merge(wd[['ds'] + regressed_features], on="ds", how="left")

# Fix the seed for the model
# set seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# make PyTorch deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [64]:
forecast_horizon = 14
regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

model = NeuralProphet(yearly_seasonality=True, 
                      weekly_seasonality=True, 
                      epochs=params['epochs'], 
                      learning_rate=params['learning_rate'], 
                      batch_size=params['batch_size'], 
                      n_forecasts = params['n_forecasts'], 
                      n_lags = params['n_lags'], 
                      ar_reg = params['ar_reg'], 
                      )
model = model.add_country_holidays(country_name="US")

for col in regressed_features:
    model.add_lagged_regressor(col, n_lags=params[col])

model.fit(rs, freq="D")
future = model.make_future_dataframe(rs, periods=forecast_horizon, n_historic_predictions=28)

for col in regressed_features:
    future[col] = pd.concat([rs[col], wd_14[col]], ignore_index=True)

forecast = model.predict(future)

rs_df = model.get_latest_forecast(forecast, include_previous_forecasts = 14)

WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(



Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/neuralprophet/data/split.py:273: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, future_df])

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.01% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.01%

Predicting: 10it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


In [65]:
df_wide= rs_df.copy()

# use the last 14 entries of 'origin-0' as the forecast
forecast_horizon = 14
origin_col = 'origin-0' 
last_14_preds = df_wide[origin_col].dropna().tail(forecast_horizon)

# compute corresponding forecast dates
last_14_dates = pd.to_datetime(df_wide['ds'].tail(forecast_horizon))

manhattan_forecast = pd.DataFrame({'ds': last_14_dates, 
                                  'yhat': np.clip(np.round(last_14_preds.astype(float)), 0, None).astype(int)})

In [66]:
manhattan_forecast

,ds,yhat
14,2026-03-14,8
15,2026-03-15,10
16,2026-03-16,17
17,2026-03-17,14
18,2026-03-18,16
19,2026-03-19,16
20,2026-03-20,12
21,2026-03-21,12
22,2026-03-22,12
23,2026-03-23,17


## Brooklyn

Need to say something about why.

In [67]:
params = dict()
params['apparent_temperature_max'] = 54
params['apparent_temperature_min'] = 18
params['snowfall_sum'] = 1
params['n_lags'] = 59 
params['epochs'] = 493 
params['learning_rate'] = 0.003214767890388168 
params['batch_size'] = 220
params['ar_reg'] = 0.5847571241076923
params['n_forecasts'] = 14 

In [68]:
rs = pd.read_csv('../scr/data/rat_sightings_data/Updated_Rat_Sightings_NYC.csv')
rs.columns
rs = rs[['Borough', 'Created Date']]


rs.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rs.columns] #apply to column headers
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']

rs = rs[rs['borough']=='BROOKLYN']



rs['created_date'] = pd.to_datetime(rs['created_date'])
rs['ds'] = rs['created_date'].dt.date
rs = rs.groupby('ds').size().reset_index(name='y')
rs['ds'] = pd.to_datetime(rs['ds'])

full_range = pd.date_range(start='2020-01-01', end=today)

# Reindex to include all dates and fill missing y with 0
rs = rs.set_index('ds').reindex(full_range, fill_value=0).rename_axis('ds').reset_index()

WARNING - (py.warnings._showwarnmsg) - /var/folders/ry/m6r2ndwd10bdv8tvww5hr2680000gn/T/ipykernel_97423/1299903892.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  rs['created_date'] = pd.to_datetime(rs['created_date'])



In [69]:
# Weather data

lat, lon = 40.7831, -73.9712
last_date = today
start = "2020-01-01"
end   = last_date.strftime("%Y-%m-%d")  # use last date of rs

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd['ds'] = pd.to_datetime(nd['date'])
    wd = nd.drop(columns=['date'])
else:
    wd = pd.DataFrame(data["daily"])
    wd["ds"] = pd.to_datetime(wd["time"])
    wd = wd.drop(columns=["time"])

wd = wd.reset_index(drop=True)
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1),
                             periods=14,
                             freq='D')

last_row = wd.iloc[-1]
wd_14 = pd.DataFrame([last_row.values] * 14, columns=wd.columns)
wd_14['ds'] = future_dates 
wd_14 = wd_14.reset_index(drop=True)

# Suppress cmdstanpy info logs
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

# set-up regressed features and final set-up for forecasting

regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']
wd["ds"] = pd.to_datetime(wd["ds"])
wd_14["ds"] = pd.to_datetime(wd_14["ds"])
rs["ds"] = pd.to_datetime(rs["ds"])
rs = rs.merge(wd[['ds'] + regressed_features], on="ds", how="left")

# Fix the seed for the model
# set seeds for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# make PyTorch deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [70]:
forecast_horizon = 14
regressed_features = ['apparent_temperature_max', 'apparent_temperature_min', 'snowfall_sum']

model = NeuralProphet(yearly_seasonality=True, 
                      weekly_seasonality=True, 
                      epochs=params['epochs'], 
                      learning_rate=params['learning_rate'], 
                      batch_size=params['batch_size'], 
                      n_forecasts = params['n_forecasts'], 
                      n_lags = params['n_lags'], 
                      ar_reg = params['ar_reg'], 
                      )
model = model.add_country_holidays(country_name="US")

for col in regressed_features:
    model.add_lagged_regressor(col, n_lags=params[col])

model.fit(rs, freq="D")
future = model.make_future_dataframe(rs, periods=forecast_horizon, n_historic_predictions=28)

for col in regressed_features:
    future[col] = pd.concat([rs[col], wd_14[col]], ignore_index=True)

forecast = model.predict(future)

rs_df = model.get_latest_forecast(forecast, include_previous_forecasts = 14)

WARNING - (NP.forecaster.fit) - When Global modeling with local normalization, metrics are displayed in normalized scale.
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.config.init_data_params) - Setting normalization to global as only one dataframe provided for training.
INFO - (NP.utils.set_auto_seasonalities) - Disabling daily seasonality. Run NeuralProphet with daily_seasonality=True to override this.
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/pytorch_lightning/trainer/setup.py:201: UserWarning: MPS available but not used. Set `accelerator` and `devices` using `Trainer(accelerator='mps', devices=1)`.
  rank_zero_warn(



Training: 0it [00:00, ?it/s]

INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.956% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
WARNING - (py.warnings._showwarnmsg) - /opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/neuralprophet/data/split.py:273: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, future_df])

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.01% of the data.
INFO - (NP.df_utils._infer_frequency) - Defined frequency is equal to major frequency - D
INFO - (NP.df_utils._infer_frequency) - Major frequency D corresponds to 99.01%

Predicting: 10it [00:00, ?it/s]

INFO - (NP.df_utils.return_df_in_original_format) - Returning df with no ID column


In [71]:
df_wide= rs_df.copy()

# use the last 14 entries of 'origin-0' as the forecast
forecast_horizon = 14
origin_col = 'origin-0' 
last_14_preds = df_wide[origin_col].dropna().tail(forecast_horizon)

# compute corresponding forecast dates
last_14_dates = pd.to_datetime(df_wide['ds'].tail(forecast_horizon))

brooklyn_forecast = pd.DataFrame({'ds': last_14_dates, 
                                  'yhat': np.clip(np.round(last_14_preds.astype(float)), 0, None).astype(int)})

In [72]:
brooklyn_forecast

,ds,yhat
14,2026-03-14,11
15,2026-03-15,13
16,2026-03-16,22
17,2026-03-17,18
18,2026-03-18,20
19,2026-03-19,21
20,2026-03-20,17
21,2026-03-21,12
22,2026-03-22,13
23,2026-03-23,22


## Staten Island

The number of rat sightings in Staten Island on any given day often is 0. Due to the sparsity of the data, we recognize that a daily forecast of two weeks might be unhelpful and so we provide in addition a weekly forecast.

1. For daily data, Prophet performed the best.

2. For weekly data, the baseline 4 year seasonal average model actually performed the best.

### Daily

In [73]:
rs = pd.read_csv('../scr/data/rat_sightings_data/Updated_Rat_Sightings_NYC.csv')
rs.columns
rs = rs[['Borough', 'Created Date']]


rs.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rs.columns] #apply to column headers
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']

rs = rs[rs['borough']=='STATEN ISLAND']

rs['created_date'] = pd.to_datetime(rs['created_date'])
rs['ds'] = rs['created_date'].dt.date
rs = rs.groupby('ds').size().reset_index(name='y')
rs['ds'] = pd.to_datetime(rs['ds'])



# Date range from 2020-01-01 to today
full_range = pd.date_range(start='2020-01-01', end=today)

# Reindex to include all dates and fill missing y with 0
rs = rs.set_index('ds').reindex(full_range, fill_value=0).rename_axis('ds').reset_index()

WARNING - (py.warnings._showwarnmsg) - /var/folders/ry/m6r2ndwd10bdv8tvww5hr2680000gn/T/ipykernel_97423/1392309216.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  rs['created_date'] = pd.to_datetime(rs['created_date'])



In [74]:
model = Prophet(changepoint_prior_scale=0.5, seasonality_prior_scale=2) # parameters chosen from 2models_statenIsland_prophet.ipynb
model.fit(rs)
future = model.make_future_dataframe(periods=14)
forecast = model.predict(future)
daily_staten_island_forecast = (forecast[['ds', 'yhat']].tail(14).assign(yhat=lambda x: x['yhat'].round().clip(lower=0).astype(int)))
daily_staten_island_forecast

,ds,yhat
2264,2026-03-14,1
2265,2026-03-15,1
2266,2026-03-16,1
2267,2026-03-17,1
2268,2026-03-18,1
2269,2026-03-19,1
2270,2026-03-20,1
2271,2026-03-21,1
2272,2026-03-22,1
2273,2026-03-23,1


### Weekly

In [75]:
rs = pd.read_csv('../scr/data/rat_sightings_data/Updated_Rat_Sightings_NYC.csv')
rs.columns
rs = rs[['Borough', 'Created Date']]


rs.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rs.columns] #apply to column headers
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']

rs = rs[rs['borough']=='STATEN ISLAND']

rs['created_date'] = pd.to_datetime(rs['created_date'])
rs['ds'] = rs['created_date'].dt.date
rs = rs.groupby('ds').size().reset_index(name='y')
rs['ds'] = pd.to_datetime(rs['ds'])

# Date range from 2020-01-01 to today
full_range = pd.date_range(start='2020-01-01', end=today)

# Reindex to include all dates and fill missing y with 0
rs = rs.set_index('ds').reindex(full_range, fill_value=0).rename_axis('ds').reset_index()

## Resample to weekly
rs_weekly = rs.set_index('ds').resample('W')['y'].mean().fillna(0).reset_index().rename(columns={'y': 'weekly_avg'})
## Rename columns for Prophet
rs_weekly.rename(columns={'created_date': 'ds', 'weekly_avg': 'y'}, inplace=True)
rs_weekly


WARNING - (py.warnings._showwarnmsg) - /var/folders/ry/m6r2ndwd10bdv8tvww5hr2680000gn/T/ipykernel_97423/216863318.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  rs['created_date'] = pd.to_datetime(rs['created_date'])



,ds,y
0,2020-01-05,0.600000
1,2020-01-12,1.285714
2,2020-01-19,1.285714
3,2020-01-26,0.571429
4,2020-02-02,1.142857
...,...,...
319,2026-02-15,1.285714
320,2026-02-22,0.285714
321,2026-03-01,0.285714
322,2026-03-08,1.714286


## Queens

In [76]:
rs = pd.read_csv('../scr/data/rat_sightings_data/Updated_Rat_Sightings_NYC.csv')
rs.columns
rs = rs[['Borough', 'Created Date']]
rs.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rs.columns] #apply to column headers
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']
rs = rs[rs['borough']=='QUEENS']
rs['created_date'] = pd.to_datetime(rs['created_date'])
rs['ds'] = rs['created_date'].dt.date
rs = rs.groupby('ds').size().reset_index(name='y')
rs['ds'] = pd.to_datetime(rs['ds'])

# Date range from 2020-01-01 to today
full_range = pd.date_range(start='2020-01-01', end=today)

# Reindex to include all dates and fill missing y with 0
rs = rs.set_index('ds').reindex(full_range, fill_value=0).rename_axis('ds').reset_index()

WARNING - (py.warnings._showwarnmsg) - /var/folders/ry/m6r2ndwd10bdv8tvww5hr2680000gn/T/ipykernel_97423/3969644909.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  rs['created_date'] = pd.to_datetime(rs['created_date'])



In [77]:
model = Prophet(changepoint_prior_scale=0.5, seasonality_prior_scale=2) # parameters chosen from 2models_statenIsland_prophet.ipynb
model.fit(rs)
future = model.make_future_dataframe(periods=14)
forecast = model.predict(future)
queens_forecast = (forecast[['ds', 'yhat']].tail(14).assign(yhat=lambda x: x['yhat'].round().clip(lower=0).astype(int)))
queens_forecast



,ds,yhat
2264,2026-03-14,3
2265,2026-03-15,3
2266,2026-03-16,8
2267,2026-03-17,8
2268,2026-03-18,7
2269,2026-03-19,7
2270,2026-03-20,6
2271,2026-03-21,3
2272,2026-03-22,3
2273,2026-03-23,8


## Bronx

In [78]:
rs = pd.read_csv('../scr/data/rat_sightings_data/Updated_Rat_Sightings_NYC.csv')
rs.columns
rs = rs[['Borough', 'Created Date']]
rs.columns = [t.partition('(')[0].strip().lower().replace(' ', '_') for t in rs.columns] #apply to column headers
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs = rs[rs['created_date']>='2020-01-01']
rs = rs[rs['borough']=='BRONX']
rs['created_date'] = pd.to_datetime(rs['created_date'])
rs['ds'] = rs['created_date'].dt.date
rs = rs.groupby('ds').size().reset_index(name='y')
rs['ds'] = pd.to_datetime(rs['ds'])

# Date range from 2020-01-01 to today
full_range = pd.date_range(start='2020-01-01', end=today)

# Reindex to include all dates and fill missing y with 0
rs = rs.set_index('ds').reindex(full_range, fill_value=0).rename_axis('ds').reset_index()

WARNING - (py.warnings._showwarnmsg) - /var/folders/ry/m6r2ndwd10bdv8tvww5hr2680000gn/T/ipykernel_97423/35266402.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  rs['created_date'] = pd.to_datetime(rs['created_date'])



In [79]:
model = Prophet(changepoint_prior_scale=0.5, seasonality_prior_scale=2) # parameters chosen from 2models_statenIsland_prophet.ipynb
model.fit(rs)
future = model.make_future_dataframe(periods=14)
forecast = model.predict(future)
bronx_forecast = (forecast[['ds', 'yhat']].tail(14).assign(yhat=lambda x: x['yhat'].round().clip(lower=0).astype(int)))
bronx_forecast

,ds,yhat
2264,2026-03-14,4
2265,2026-03-15,4
2266,2026-03-16,8
2267,2026-03-17,8
2268,2026-03-18,8
2269,2026-03-19,8
2270,2026-03-20,6
2271,2026-03-21,4
2272,2026-03-22,3
2273,2026-03-23,8


# Collected Forecasts

In [80]:
citywide_forecast

,ds,yhat
14,2026-03-14,14
15,2026-03-15,23
16,2026-03-16,48
17,2026-03-17,44
18,2026-03-18,44
19,2026-03-19,42
20,2026-03-20,35
21,2026-03-21,18
22,2026-03-22,22
23,2026-03-23,49


In [81]:
manhattan_forecast

,ds,yhat
14,2026-03-14,8
15,2026-03-15,10
16,2026-03-16,17
17,2026-03-17,14
18,2026-03-18,16
19,2026-03-19,16
20,2026-03-20,12
21,2026-03-21,12
22,2026-03-22,12
23,2026-03-23,17


In [82]:
brooklyn_forecast

,ds,yhat
14,2026-03-14,11
15,2026-03-15,13
16,2026-03-16,22
17,2026-03-17,18
18,2026-03-18,20
19,2026-03-19,21
20,2026-03-20,17
21,2026-03-21,12
22,2026-03-22,13
23,2026-03-23,22


In [83]:
daily_staten_island_forecast

,ds,yhat
2264,2026-03-14,1
2265,2026-03-15,1
2266,2026-03-16,1
2267,2026-03-17,1
2268,2026-03-18,1
2269,2026-03-19,1
2270,2026-03-20,1
2271,2026-03-21,1
2272,2026-03-22,1
2273,2026-03-23,1


In [84]:
# weekly_staten_island_forecast

In [85]:
queens_forecast

,ds,yhat
2264,2026-03-14,3
2265,2026-03-15,3
2266,2026-03-16,8
2267,2026-03-17,8
2268,2026-03-18,7
2269,2026-03-19,7
2270,2026-03-20,6
2271,2026-03-21,3
2272,2026-03-22,3
2273,2026-03-23,8


In [86]:
bronx_forecast

,ds,yhat
2264,2026-03-14,4
2265,2026-03-15,4
2266,2026-03-16,8
2267,2026-03-17,8
2268,2026-03-18,8
2269,2026-03-19,8
2270,2026-03-20,6
2271,2026-03-21,4
2272,2026-03-22,3
2273,2026-03-23,8


In [87]:
dfs = {"Citywide": citywide_forecast,
    "Manhattan": manhattan_forecast,
    "Brooklyn": brooklyn_forecast,
    "Daily Staten Island": daily_staten_island_forecast,
    "Queens": queens_forecast,
    "Bronx": bronx_forecast,
    #"Weekly Staten Island": weekly_staten_island_forecast
}

# Prepare each dataframe
processed = []
for name, df in dfs.items():
    temp = df[['ds', 'yhat']].copy()
    temp = temp.set_index('ds')
    temp = temp.rename(columns={'yhat': name})
    processed.append(temp)

# Combine them side by side
final_df = pd.concat(processed, axis=1)

# Reset index so ds becomes a column again
final_df = final_df.reset_index().rename(columns={'ds': 'Date'})

borough_columns = ["Manhattan", "Brooklyn", "Daily Staten Island", "Queens", "Bronx"]
existing_boroughs = [col for col in borough_columns if col in final_df.columns]
final_df["Summed Citywide"] = final_df[existing_boroughs].sum(axis=1)

final_df

,Date,Citywide,Manhattan,Brooklyn,Daily Staten Island,Queens,Bronx,Summed Citywide
0,2026-03-14,14,8,11,1,3,4,27
1,2026-03-15,23,10,13,1,3,4,31
2,2026-03-16,48,17,22,1,8,8,56
3,2026-03-17,44,14,18,1,8,8,49
4,2026-03-18,44,16,20,1,7,8,52
5,2026-03-19,42,16,21,1,7,8,53
6,2026-03-20,35,12,17,1,6,6,42
7,2026-03-21,18,12,12,1,3,4,32
8,2026-03-22,22,12,13,1,3,3,32
9,2026-03-23,49,17,22,1,8,8,56
